# Multivariable Linear Regression for ROE Prediction

In [2]:
# Importing pckages and dataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn import set_config

# Output dataframes instead of arrays
set_config(transform_output="pandas")

**Our Objective is to predict After Tax Return on Equity (ROE) for companies listed in New York Stock Exchange based on its financial matrics**

In [3]:
# Loading the dataset

df = pd.read_csv('../nyse-prediction/data/raw/fundamentals.csv')

# Display the first few rows to inspect the data
print("First five rows of the dataset:")
print(df.head())

First five rows of the dataset:
   Unnamed: 0 Ticker Symbol Period Ending  Accounts Payable  \
0           0           AAL    2012-12-31      3.068000e+09   
1           1           AAL    2013-12-31      4.975000e+09   
2           2           AAL    2014-12-31      4.668000e+09   
3           3           AAL    2015-12-31      5.102000e+09   
4           4           AAP    2012-12-29      2.409453e+09   

   Accounts Receivable  Add'l income/expense items  After Tax ROE  \
0         -222000000.0               -1.961000e+09           23.0   
1          -93000000.0               -2.723000e+09           67.0   
2         -160000000.0               -1.500000e+08          143.0   
3          352000000.0               -7.080000e+08          135.0   
4          -89482000.0                6.000000e+05           32.0   

   Capital Expenditures  Capital Surplus  Cash Ratio  ...  \
0         -1.888000e+09     4.695000e+09        53.0  ...   
1         -3.114000e+09     1.059200e+10        75.0

### Training, evaluating, and tuning the model

#### **Step 1:** Split the dataset into test and train.

In [4]:
# Handle missing values 
nyse = df.dropna()  # drop rows with missing values

# Select the relevant features and target
X = nyse[['Accounts Payable', 'Accounts Receivable', "Add'l income/expense items",  
        'Capital Expenditures', 'Capital Surplus', 'Cash Ratio','Pre-Tax ROE','Profit Margin','Pre-Tax Margin','Misc. Stocks',
        'Changes in Inventories','Inventory','Other Financing Activities','Current Ratio',
        'Gross Margin','Income Tax','Minority Interest','Net Income','Net Income Applicable to Common Shareholders',
        'Operating Margin','Equity Earnings/Loss Unconsolidated Subsidiary']]  # Selected Features 
y = nyse['After Tax ROE']  # Target variable

# Split the data into training and testing sets
nyse_train, nyse_test = train_test_split(nyse, train_size=0.75, random_state=42)

#### **Step 2:** Fit the Multivariable Linear Regression Model.

The equation for the multivariable regression is:

$$
\text{After Tax ROE} = b_0 + b_1 \times (\text{Accounts Payable}) + b_2 \times (\text{Accounts Receivable}) + b_3 \times (\text{Add'l income/expense items})+ b_4 \times (\text{Capital Expenditures})+ \\ 
b_5 \times (\text{Capital Surplus})+ b_6 \times (\text{Cash Ratio})+ b_7 \times (\text{Pre-Tax ROE})+ b_8 \times (\text{Profit Margin})+ b_9 \times (\text{Pre-Tax Margin})\\+ b_{10} \times (\text{Misc. Stocks})+ b_{11} \times (\text{Changes in Inventories})+ b_{12} \times (\text{Inventory})+ b_{13} \times (\text{Other Financing Activities})\\ +b_{14} \times (\text{Current Ratio})+b_{15} \times (\text{Gross Margin})+b_{16} \times (\text{Income Tax})+b_{17} \times (\text{Minority Interest})+b_{18} \times (\text{Net Income})\\+b_{19} \times (\text{Net Income Applicable to Common Shareholders})+b_{20} \times (\text{Operating Margin})\\ +b_{21} \times (\text{Equity Earnings/Loss Unconsolidated Subsidiary})
$$

where:

- $b_0$ is the Return on equity when all the predictors are 0 (the intercept).
- $b_1$ is how much the ROE increases/decreases for each unit increase in Accounts Payable (the slope for Accounts Payable).
- $b_2$ is how much the ROE increases/decrease for each additional income/ expense item (the slope for Accounts Receivable) and similar for other predictors.

In [5]:
# Multivariable Linear Regression
mlm = LinearRegression()

mlm.fit(
    nyse_train[['Accounts Payable', 'Accounts Receivable', "Add'l income/expense items",  
        'Capital Expenditures', 'Capital Surplus', 'Cash Ratio','Pre-Tax ROE','Profit Margin','Pre-Tax Margin','Misc. Stocks',
        'Changes in Inventories','Inventory','Other Financing Activities','Current Ratio',
        'Gross Margin','Income Tax','Minority Interest','Net Income','Net Income Applicable to Common Shareholders',
        'Operating Margin','Equity Earnings/Loss Unconsolidated Subsidiary']],  # Selected Features
    nyse_train['After Tax ROE']  # Target variable
)

LinearRegression()

#### **Step 3:** Make predictions on the test data set to assess the quality of our model.


In [6]:
# Predict ROE using the multivariable linear regression model (mlm)

nyse_test["predicted"] = mlm.predict(nyse_test[['Accounts Payable', 'Accounts Receivable', "Add'l income/expense items",  
        'Capital Expenditures', 'Capital Surplus', 'Cash Ratio','Pre-Tax ROE','Profit Margin','Pre-Tax Margin','Misc. Stocks',
        'Changes in Inventories','Inventory','Other Financing Activities','Current Ratio',
        'Gross Margin','Income Tax','Minority Interest','Net Income','Net Income Applicable to Common Shareholders',
        'Operating Margin','Equity Earnings/Loss Unconsolidated Subsidiary']])

# Calculate RMSPE for the multivariable model.
lm_mult_test_RMSPE = mean_squared_error(
    y_true=nyse_test["After Tax ROE"],
    y_pred=nyse_test["predicted"]
)**(1/2)

lm_mult_test_RMSPE

8.509317577544918

RMSPE of 8.51% indicates that the model predicting ROE for NYSE-listed companies is fairly accurate, but the error margin means there’s an average deviation of 8.51% between actual and predicted ROE values. Predicting ROE with 8.51% accuracy may be acceptable for broader market insights, but it could be considered high for high-frequency trading or risk management.

In [7]:
mlm.intercept_

3.715478194982687

In [8]:
mlm.coef_

array([ 1.26183811e-10, -2.64581795e-10,  8.23938023e-10, -7.34226097e-11,
        8.73823376e-11,  3.59515319e-02,  6.76468361e-01,  1.30938473e+00,
       -1.01352133e+00,  2.62814015e-10, -2.88962253e-11, -3.89763100e-11,
        6.45321030e-10, -3.06016834e-02, -4.64749761e-02, -2.76650555e-09,
       -2.83036260e-10, -8.73233262e-09,  9.23890972e-09,  2.47303278e-01,
        2.24170194e-09])

Positive coefficients for Accounts payable, Add'l income/expense items, Capital Surplus, Cash Ratio, Pre-Tax ROE, Profit Margin, Misc. Stocks, Other Financing Activities, Net Income Applicable to Common Shareholders, Operating Margin, Equity Earnings/Loss Unconsolidated Subsidiary indicate that an increase in these predictors is associated with higher ROE.